# 08 — Can the J-lens's top tokens tell a lie from an honest answer?

04d showed the true answer is linearly decodable from the 2560-d residual under D. This
notebook asks a smaller, readable question with the lens back in the loop: at the answer
slot, do the tokens the J-lens decodes at layer ℓ separate the forward pass where the
model **lied** from the ones where it did not — and which tokens carry it?

**The measurement, stated once** ([spec](../misc/exp4_spec.md), PLAN2 §13):

> Rows are every v2 belief item under H, C1 and D. Label = *this is the D row of an item
> the model lied on*. Features = the J-lens log-probs of the top-K tokens at layer ℓ,
> vocabulary built inside each leave-one-pair-out fold. The **primary** fit removes every
> Yes/No spelling from the vocabulary; the full-vocabulary fit sits beside it as the thing
> being controlled for. Null = labels shuffled within each item's rows. Then fit on all of
> v2 and read the alleged arm unchanged.

Two confounds this design is built around: the lying set is 37 No-true / 16 Yes-true, so
the emitted token alone is worth ~0.7 on the raw label (hence the mask, and the per-arm
split); and D differs from H in prompt text, not only in lying (hence C1 rows and the
honest D rows as negatives).

| | |
|---|---|
| instrument | [`tokenfeat.py`](../src/nandaproj/tokenfeat.py): top-K store, fold-internal vocabulary, answer mask, within-item null. Numpy, tested locally |
| GPU | section 1 only, ~580 lens readouts. Everything after reloads from `results/` |
| reads | `slot_emitted_{,alleged_}*.json` from 04d for the lying mask |
| writes | `jtoken_topk_*.npz`, `jtoken_topk_alleged_*.npz`, `jtoken_classifier_summary_*.json` |


In [ ]:
# --- which model ----------------------------------------------------------
# Run this BEFORE the header cell; `get_model_config()` reads the env var at
# call time. Do not run this in a kernel that still holds another 4B copy --
# two do not fit beside activations on a 24 GB 4090.
import gc, os

os.environ["NANDA_PRESET"] = "target"   # debug(270m) | main(1b) | target(4b) | escalate(12b)

for _name in ("reader", "model_jlens", "lens", "model"):
    globals().pop(_name, None)
gc.collect()
try:
    import torch
    torch.cuda.empty_cache()
    print("free VRAM:", round(torch.cuda.mem_get_info()[0] / 2**30, 1), "GiB")
except (ImportError, RuntimeError):
    pass


In [ ]:
# --- standard header ------------------------------------------------------
%load_ext autoreload
%autoreload 2

import sys, pathlib
for p in ("/workspace/NandaProj", ".."):
    if p not in sys.path:
        sys.path.insert(0, p)

import json

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from nandaproj import config, items, lens_readout, probe, tokenfeat, viz

cfg = config.get_model_config()
config.ensure_dirs()
print("preset:", cfg.name, "|", cfg.n_params, "|", cfg.dtype)
print("device:", config.get_device())


In [ ]:
tok = AutoTokenizer.from_pretrained(cfg.name, cache_dir=str(config.HF_CACHE))
model = AutoModelForCausalLM.from_pretrained(
    cfg.name,
    cache_dir=str(config.HF_CACHE),
    dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

# The lens is used here, unlike 04d: every feature is a J-lens (or logit-lens)
# token probability from `reader.readout`.
reader = lens_readout.Reader.load(model, tok, cfg)
print(reader.describe())


## 1. Capture — the only GPU section

All 100 v2 belief items × {H, C1, D} and the 141 alleged-arm items × {H, D}: one
`reader.readout` each, keeping the top-200 token ids and log-probs at every fitted layer
for the J-lens and, from the same call, the logit lens. About 580 readouts, two lens
passes each, so 10–20 minutes. Written to `results/` after every item.

Storing 200 rather than the full vocabulary keeps the file at a few MB; the classifier
uses at most the top-50, and a token outside a row's 200 takes that row's floor.


In [ ]:
# --- 1: capture the top-200 per row, both lenses ---------------------------------
BANK = items.load_bank()                      # data/deception_bank_export_v2.json
BELIEF = [i for i in BANK if not i.is_no_belief]
assert len(BELIEF) == 100, "expected the v2 bank: 100 belief items"
CONDS = ("H", "C1", "D")
K_STORE = 200

TOPK_NPZ = config.RESULTS / f"jtoken_topk_{cfg.lens_id}.npz"
ALG_NPZ = config.RESULTS / f"jtoken_topk_alleged_{cfg.lens_id}.npz"
ALLEGED_JSON = config.DATA / "alleged_arm_final.json"

if TOPK_NPZ.exists():
    print(f"v2 store already on disk: {TOPK_NPZ.name} (delete to recapture)")
else:
    print(items.report(items.validate(tok, BELIEF)))
    tokenfeat.capture_topk(reader, BELIEF, CONDS, k=K_STORE, save_to=TOPK_NPZ)
    print(f"{TOPK_NPZ.name}: {TOPK_NPZ.stat().st_size / 1e6:.1f} MB")

if ALG_NPZ.exists():
    print(f"alleged store already on disk: {ALG_NPZ.name} (delete to recapture)")
else:
    ALG_ITEMS = [i for i in items.load(ALLEGED_JSON, conditions=None) if not i.is_no_belief]
    print(items.report(items.validate(tok, ALG_ITEMS)))
    tokenfeat.capture_topk(reader, ALG_ITEMS, ("H", "D"), k=K_STORE, save_to=ALG_NPZ)
    print(f"{ALG_NPZ.name}: {ALG_NPZ.stat().st_size / 1e6:.1f} MB")

# Read one back: the top-5 J-lens tokens at an upper layer of one D row, as a sanity
# check that position -1 is the answer slot and not the 512th token.
_s = tokenfeat.TopK.load(TOPK_NPZ)
_r = int(_s.rows("D")[0]); _l = _s.layers[-1]
_ids, _lp = _s.at("j", _l)
print(f"\n{_s.item_ids[_r]} D, J-lens L{_l} top-5:",
      [(tok.decode([int(t)]), round(float(np.exp(p)), 3)) for t, p in zip(_ids[_r, :5], _lp[_r, :5])])


## 2. Features, CV, null — CPU only

Rows: every v2 belief item × {H, C1, D}. Label: *this row is the D pass of an item the model
lied on* (04d's `LYING` mask). H, C1 and the D rows where the model told the truth are all
label 0; those honest D rows are what stop "the persona is present" from being enough.

Per layer, per fold: the feature vocabulary is the union of the **training** rows' top-K
J-lens tokens, minus every yes/no spelling for the masked fit; feature = the row's log-prob
of each token, floored at the row's 200th. Standardise on the training rows, L2 logistic,
leave-one-pair-out. The logit-lens twin is the same pipeline on the same readouts.

Metric is **balanced accuracy**: 53 of 300 rows are lies, so plain accuracy is 0.82 for
never saying "lie". The null shuffles labels within each item's three rows.


In [15]:
# --- 2a: rows, labels, groups, the answer-token mask ------------------------------
# Runs without the model: needs only the store, 04d's emitted file, and a tokenizer.
STORE = tokenfeat.TopK.load(TOPK_NPZ)
EMIT = json.loads((config.RESULTS / f"slot_emitted_{cfg.lens_id}.json").read_text())
BY_ID = {i.item_id: i for i in items.load_bank().items}
LAYERS = STORE.layers
ROW_ITEM, ROW_COND = np.asarray(STORE.item_ids), np.asarray(STORE.conditions)

# Truth from the bank's declared polarity; "lied" from 04d's emitted probabilities,
# exactly as 04d's LYING mask: honest under H and wrong under D. No new gating.
truth = lambda i: BY_ID[i].meta["polarity"] == "Yes"
said = lambda c, i: EMIT[c][i]["p_yes"] > EMIT[c][i]["p_no"]
LYING_ITEM = {i: (said("H", i) == truth(i)) and (said("D", i) != truth(i)) for i in set(ROW_ITEM)}

Y = np.array([(c == "D") and LYING_ITEM[i] for i, c in zip(ROW_ITEM, ROW_COND)])   # this row is a lie
ROW_TRUTH = np.array([truth(i) for i in ROW_ITEM])
GROUPS = [BY_ID[i].meta["pair_id"] for i in ROW_ITEM]

if "tok" not in globals():
    tok = AutoTokenizer.from_pretrained(cfg.name, cache_dir=str(config.HF_CACHE))
BANNED = tokenfeat.answer_token_ids(tok)          # full-vocab scan, every yes/no spelling

# Every fold and every null draw of a layer is one batched fit (tokenfeat.cv_predict_many);
# on the box that is the GPU. 2b checks it against the per-fold reference at one layer.
DEVICE = tokenfeat.cuda_if_available()
print(f"batched fits on: {DEVICE or 'cpu (numpy)'}")

print(f"{len(Y)} rows = {len(set(ROW_ITEM))} items x {sorted(set(map(str, ROW_COND)))}, "
      f"{len(set(GROUPS))} pairs, {len(LAYERS)} layers, top-{STORE.k} stored per row")
print(f"lie rows: {int(Y.sum())}  ({int((Y & ~ROW_TRUTH).sum())} No-true, {int((Y & ROW_TRUTH).sum())} Yes-true)"
      f"   honest D rows: {int(((ROW_COND == 'D') & ~Y).sum())}   H/C1 rows: {int((ROW_COND != 'D').sum())}")
print(f"\nanswer-token mask: {len(BANNED)} ids -> {[tok.decode([i]) for i in BANNED]}")

METRICS = ("balanced", "tpr", "tnr", "tpr_no_true", "tpr_yes_true", "tnr_honest_D",
           "balanced_no_true", "balanced_yes_true")

def _rate(mask, pred):
    return float(pred[mask].mean()) if mask.any() else float("nan")

def score(pred, y, row_truth, row_cond):
    """Balanced accuracy over all rows, plus the pieces that say who carried it.

    `balanced_no_true` / `balanced_yes_true` are balanced accuracies computed
    within one polarity arm, so a classifier reading the answer's polarity
    (which is confounded with the label on this bank) scores 0.5 on both.
    """
    pred, y = np.asarray(pred).astype(bool), np.asarray(y).astype(bool)
    honest_D = (row_cond == "D") & ~y
    within = lambda arm: (_rate(y & arm, pred) + _rate(~y & arm, ~pred)) / 2
    return {"balanced": tokenfeat.balanced_accuracy(pred, y),
            "tpr": _rate(y, pred), "tnr": _rate(~y, ~pred),
            "tpr_no_true": _rate(y & ~row_truth, pred), "tpr_yes_true": _rate(y & row_truth, pred),
            "tnr_honest_D": _rate(honest_D, ~pred),
            "balanced_no_true": within(~row_truth), "balanced_yes_true": within(row_truth)}


batched fits on: cuda
300 rows = 100 items x ['C1', 'D', 'H'], 50 pairs, 33 layers, top-200 stored per row
lie rows: 53  (37 No-true, 16 Yes-true)   honest D rows: 47   H/C1 rows: 200

answer-token mask: 12 ids -> [' no', 'no', ' No', 'No', 'yes', 'NO', ' Yes', ' NO', 'Yes', ' yes', 'YES', ' YES']


In [16]:
# --- 2b: leave-one-pair-out sweep, every layer ----------------------------------
K_PRIMARY, K_BAND = 20, (5, 50)
CONFIGS = ([("j", K_PRIMARY, True), ("j", K_PRIMARY, False),
            ("l", K_PRIMARY, True), ("l", K_PRIMARY, False)]
           + [("j", k, True) for k in K_BAND])

# One-off check on the box: the batched (GPU) fit and the per-fold reference must
# give identical predictions. Done at one layer of the primary configuration.
_l0 = LAYERS[len(LAYERS) // 2]
_ids, _lp = STORE.at("j", _l0)
_ref = tokenfeat.cv_predict(_ids, _lp, Y, GROUPS, K_PRIMARY, BANNED)
_got = tokenfeat.cv_predict_many(_ids, _lp, Y, GROUPS, K_PRIMARY, BANNED, device=DEVICE)[0]
_dis = int((_ref != _got).sum())
assert _dis <= 1, f"batched fit disagrees with the per-fold reference on {_dis} rows at L{_l0}"
print(f"batched fit == per-fold reference at L{_l0}: {_dis} disagreements of {len(Y)}")

CV = {}
for lens, k, masked in CONFIGS:
    out = {m: [] for m in METRICS}
    for l in lens_readout._progress(LAYERS, desc=f"{lens} K={k} {'masked' if masked else 'full'}"):
        ids, lp = STORE.at(lens, l)
        pred = tokenfeat.cv_predict_many(ids, lp, Y, GROUPS, k, BANNED if masked else (), device=DEVICE)[0]
        s = score(pred, Y, ROW_TRUTH, ROW_COND)
        for m in METRICS:
            out[m].append(s[m])
    CV[(lens, k, masked)] = out

viz.series_line(
    LAYERS,
    {"J-lens, answer tokens masked": CV[("j", K_PRIMARY, True)]["balanced"],
     "J-lens, full vocab": CV[("j", K_PRIMARY, False)]["balanced"],
     "logit lens, answer tokens masked": CV[("l", K_PRIMARY, True)]["balanced"],
     "logit lens, full vocab": CV[("l", K_PRIMARY, False)]["balanced"]},
    y_range=(0, 1), title=f"Lie-row classifier, leave-one-pair-out, K={K_PRIMARY}",
    xaxis="layer", yaxis="balanced accuracy",
).show()

viz.series_line(
    LAYERS,
    {f"J masked K={k}": CV[("j", k, True)]["balanced"] for k in (K_BAND[0], K_PRIMARY, K_BAND[1])},
    y_range=(0, 1), title="Sensitivity to K (J-lens, answer tokens masked)",
    xaxis="layer", yaxis="balanced accuracy",
).show()

jm = CV[("j", K_PRIMARY, True)]
print(f"{'layer':>5} {'balanced':>8} {'TPR No-true':>11} {'TPR Yes-true':>12} {'TNR honest D':>12} "
      f"{'bal No-arm':>10} {'bal Yes-arm':>11}   (J-lens, masked, K=20)")
for l in LAYERS[::2]:
    i = LAYERS.index(l)
    print(f"{l:>5} {jm['balanced'][i]:>8.2f} {jm['tpr_no_true'][i]:>11.2f} "
          f"{jm['tpr_yes_true'][i]:>12.2f} {jm['tnr_honest_D'][i]:>12.2f} "
          f"{jm['balanced_no_true'][i]:>10.2f} {jm['balanced_yes_true'][i]:>11.2f}")
print("\nA balanced score carried by one polarity arm with the other near 0.5 is a token result,\n"
      "whatever the vocabulary mask says: read the two within-arm columns before the curve.\n"
      "TNR on honest D rows is the only column that separates 'told to lie' from 'lied'; 2d\n"
      "makes that the whole comparison.")


batched fit == per-fold reference at L16: 0 disagreements of 300


j K=20 masked:   0%|          | 0/33 [00:00<?, ?it/s]

j K=20 full:   0%|          | 0/33 [00:00<?, ?it/s]

l K=20 masked:   0%|          | 0/33 [00:00<?, ?it/s]

l K=20 full:   0%|          | 0/33 [00:00<?, ?it/s]

j K=5 masked:   0%|          | 0/33 [00:00<?, ?it/s]

j K=50 masked:   0%|          | 0/33 [00:00<?, ?it/s]

layer balanced TPR No-true TPR Yes-true TNR honest D bal No-arm bal Yes-arm   (J-lens, masked, K=20)
    0     0.59        0.19         0.31         0.83       0.58        0.63
    2     0.55        0.16         0.06         0.87       0.57        0.51
    4     0.50        0.00         0.00         1.00       0.50        0.50
    6     0.67        0.30         0.56         0.81       0.64        0.75
    8     0.74        0.46         0.69         0.77       0.71        0.81
   10     0.83        0.73         0.69         0.68       0.86        0.80
   12     0.79        0.57         0.81         0.68       0.77        0.86
   14     0.87        0.84         0.81         0.57       0.90        0.85
   16     0.91        0.92         0.75         0.74       0.95        0.84
   18     0.92        0.92         0.88         0.70       0.95        0.90
   20     0.92        0.89         0.88         0.70       0.94        0.89
   22     0.86        0.84         0.69         0.64       0.91

In [17]:
# --- 2c: the null, primary configuration only ------------------------------------
# Labels shuffled within each item's three rows: a lying item keeps one positive
# row, but which condition carries it is random. All NULL_N draws and all 50 folds
# of a layer are one batched fit, so this is seconds per layer on the GPU. NULL_LAYERS
# is a knob in case it is not; everything downstream looks the null up by layer.
import time

NULL_N = 200
NULL_LAYERS = list(LAYERS)
NULL, NULL_P95 = {}, {}
t0 = time.time()
for l in lens_readout._progress(NULL_LAYERS, desc="null (within-item shuffle)"):
    rng = np.random.default_rng(l)
    Ys = np.stack([Y] + [tokenfeat.shuffle_within_items(Y, STORE.item_ids, rng) for _ in range(NULL_N)])
    ids, lp = STORE.at("j", l)
    preds = tokenfeat.cv_predict_many(ids, lp, Ys, GROUPS, K_PRIMARY, BANNED, device=DEVICE)
    NULL[l] = np.array([tokenfeat.balanced_accuracy(p, y) for p, y in zip(preds[1:], Ys[1:])])
    NULL_P95[l] = float(np.percentile(NULL[l], 95))
    # Row 0 is the observed labelling: must reproduce 2b's number for this layer.
    obs = tokenfeat.balanced_accuracy(preds[0], Y)
    assert abs(obs - CV[("j", K_PRIMARY, True)]["balanced"][LAYERS.index(l)]) < 1e-6
print(f"{len(NULL_LAYERS)} layers x {NULL_N} draws in {time.time() - t0:.0f} s")

viz.series_line(
    NULL_LAYERS,
    {"J-lens, answer tokens masked (K=20)": [CV[("j", K_PRIMARY, True)]["balanced"][LAYERS.index(l)]
                                             for l in NULL_LAYERS],
     "null p95 (within-item shuffle)": [NULL_P95[l] for l in NULL_LAYERS]},
    y_range=(0, 1), title="Primary curve against its null",
    xaxis="layer", yaxis="balanced accuracy",
).show()
above = [l for l in NULL_LAYERS if CV[("j", K_PRIMARY, True)]["balanced"][LAYERS.index(l)] > NULL_P95[l]]
print(f"masked J-lens curve above null p95 at {len(above)}/{len(NULL_LAYERS)} layers tested: {above}")
print("Remember what this null cannot rule out: a classifier that only reads the lie\n"
      "instruction in the context clears it at balanced 0.90. See 2d.")


null (within-item shuffle):   0%|          | 0/33 [00:00<?, ?it/s]

33 layers x 200 draws in 10 s


masked J-lens curve above null p95 at 32/33 layers tested: [0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
Remember what this null cannot rule out: a classifier that only reads the lie
instruction in the context clears it at balanced 0.90. See 2d.


In [18]:
# --- 2d: D rows only -- lied vs did not, under the identical prompt template ------
# The balanced score above can be cleared by a classifier that only detects the lie
# instruction in the context: TPR 1.0 on the 53 lie rows, TNR 0.81 on negatives of
# which just 47 carry the D prompt, balanced 0.90. The 47 honest D rows are the only
# rows that separate "told to lie" from "lied". So: D rows alone, 53 vs 47, same
# persona and instruction on both sides, with its own null (label permutation across
# rows; within-item shuffling is meaningless with one D row per item).
from collections import Counter

D_ROWS = np.flatnonzero(ROW_COND == "D")
Y_D, TRUTH_D = Y[D_ROWS], ROW_TRUTH[D_ROWS]
GROUPS_D = [GROUPS[i] for i in D_ROWS]
COND_D = ROW_COND[D_ROWS]
TIER_D = np.array([BY_ID[ROW_ITEM[i]].tier for i in D_ROWS])
print(f"D rows: {len(D_ROWS)}  lied {int(Y_D.sum())} ({int((Y_D & ~TRUTH_D).sum())} No-true, "
      f"{int((Y_D & TRUTH_D).sum())} Yes-true)   honest {int((~Y_D).sum())} "
      f"({int((~Y_D & ~TRUTH_D).sum())} No-true, {int((~Y_D & TRUTH_D).sum())} Yes-true)")
print("Polarity is confounded with the label here (the model lies with 'Yes' far more\n"
      "readily), so the within-arm balanced columns below are the ones to read.")

# Item-selection check: do the items the model refused to lie on differ in kind?
for field in ("tier", "harm"):
    lie_c = Counter(getattr(BY_ID[ROW_ITEM[i]], field) for i in D_ROWS if Y[i])
    hon_c = Counter(getattr(BY_ID[ROW_ITEM[i]], field) for i in D_ROWS if not Y[i])
    print(f"\n{field:<22} {'lied':>5} {'honest':>7}")
    for k in sorted(set(lie_c) | set(hon_c), key=str):
        print(f"  {str(k):<20} {lie_c[k]:>5} {hon_c[k]:>7}")

# Two populations: all D rows, and D rows without the tier the model will not lie
# on at all (world_fact: 1 lied / 19 honest in the v2 gate), which otherwise makes
# "honest" nearly synonymous with "world fact" and the classifier a tier detector.
POPS = {"all D rows": np.arange(len(D_ROWS)),
        "D rows minus world_fact": np.flatnonzero(TIER_D != "world_fact")}
for name, sel in POPS.items():
    print(f"\n{name}: {len(sel)} rows, lied {int(Y_D[sel].sum())}, honest {int((~Y_D[sel]).sum())}")

D_CONFIGS = [("j", True), ("j", False), ("l", True)]
CV_D, NULL_D_P95 = {}, {}
for pop, sel in POPS.items():
    ids_all, lp_all = None, None
    y_p, g_p, t_p, c_p = Y_D[sel], [GROUPS_D[i] for i in sel], TRUTH_D[sel], COND_D[sel]
    rows = D_ROWS[sel]
    for lens, masked in D_CONFIGS:
        out = {m: [] for m in METRICS}
        for l in lens_readout._progress(LAYERS, desc=f"{pop}: {lens} {'masked' if masked else 'full'}"):
            ids, lp = STORE.at(lens, l)
            pred = tokenfeat.cv_predict_many(ids[rows], lp[rows], y_p, g_p, K_PRIMARY,
                                             BANNED if masked else (), device=DEVICE)[0]
            s = score(pred, y_p, t_p, c_p)
            for m in METRICS:
                out[m].append(s[m])
        CV_D[(pop, lens, masked)] = out
    NULL_D_P95[pop] = {}
    for l in lens_readout._progress(NULL_LAYERS, desc=f"{pop}: null (label permutation)"):
        rng = np.random.default_rng(1000 + l)
        Ys = np.stack([y_p] + [rng.permutation(y_p) for _ in range(NULL_N)])
        ids, lp = STORE.at("j", l)
        preds = tokenfeat.cv_predict_many(ids[rows], lp[rows], Ys, g_p, K_PRIMARY, BANNED, device=DEVICE)
        draws = [tokenfeat.balanced_accuracy(p, y) for p, y in zip(preds[1:], Ys[1:])]
        NULL_D_P95[pop][l] = float(np.percentile(draws, 95))

for pop in POPS:
    viz.series_line(
        LAYERS,
        {"J-lens, answer tokens masked": CV_D[(pop, "j", True)]["balanced"],
         "J-lens, full vocab": CV_D[(pop, "j", False)]["balanced"],
         "logit lens, answer tokens masked": CV_D[(pop, "l", True)]["balanced"],
         "null p95 (label permutation)": [NULL_D_P95[pop].get(l, np.nan) for l in LAYERS]},
        y_range=(0, 1), title=f"{pop}: lied vs honest, leave-one-pair-out",
        xaxis="layer", yaxis="balanced accuracy",
    ).show()
    jd = CV_D[(pop, "j", True)]
    print(f"\n{pop} (J-lens, masked, K=20)")
    print(f"{'layer':>5} {'balanced':>8} {'null p95':>8} {'bal No-arm':>10} {'bal Yes-arm':>11} "
          f"{'TPR No-true':>11} {'TPR Yes-true':>12} {'TNR honest':>10}")
    for l in LAYERS[::2]:
        i = LAYERS.index(l)
        print(f"{l:>5} {jd['balanced'][i]:>8.2f} {NULL_D_P95[pop].get(l, float('nan')):>8.2f} "
              f"{jd['balanced_no_true'][i]:>10.2f} {jd['balanced_yes_true'][i]:>11.2f} "
              f"{jd['tpr_no_true'][i]:>11.2f} {jd['tpr_yes_true'][i]:>12.2f} {jd['tnr'][i]:>10.2f}")
    above_d = [l for l in NULL_LAYERS if jd["balanced"][LAYERS.index(l)] > NULL_D_P95[pop][l]]
    print(f"above null p95 at {len(above_d)}/{len(NULL_LAYERS)} layers tested: {above_d}")
print("\nThe D-only, world_fact-excluded, within-arm numbers are the ones that say whether\n"
      "anything beyond the instruction, the tier, or the answer's polarity was read.")


D rows: 100  lied 53 (37 No-true, 16 Yes-true)   honest 47 (13 No-true, 34 Yes-true)
Polarity is confounded with the label here (the model lies with 'Yes' far more
readily), so the within-arm balanced columns below are the ones to read.

tier                    lied  honest
  product_fault           21       7
  safety_fact             20       8
  social_white_lie        11      13
  world_fact               1      19

harm                    lied  honest
  None                     1      19
  high                    31      11
  low                     21      17

all D rows: 100 rows, lied 53, honest 47

D rows minus world_fact: 80 rows, lied 52, honest 28


all D rows: j masked:   0%|          | 0/33 [00:00<?, ?it/s]

all D rows: j full:   0%|          | 0/33 [00:00<?, ?it/s]

all D rows: l masked:   0%|          | 0/33 [00:00<?, ?it/s]

all D rows: null (label permutation):   0%|          | 0/33 [00:00<?, ?it/s]

...

D rows minus world_fact: j full:   0%|          | 0/33 [00:00<?, ?it/s]

D rows minus world_fact: l masked:   0%|          | 0/33 [00:00<?, ?it/s]

D rows minus world_fact: null (label permutation):   0%|          | 0/33 [00:00<?, ?it/s]


all D rows (J-lens, masked, K=20)
layer balanced null p95 bal No-arm bal Yes-arm TPR No-true TPR Yes-true TNR honest
    0     0.60     0.59       0.63        0.64        0.65         0.81       0.51
    2     0.72     0.61       0.77        0.75        0.70         0.88       0.68
    4     0.61     0.58       0.68        0.64        0.68         0.88       0.49
    6     0.67     0.60       0.72        0.70        0.76         0.94       0.53
    8     0.70     0.60       0.75        0.72        0.73         0.88       0.62
   10     0.73     0.59       0.75        0.75        0.81         0.88       0.64
   12     0.72     0.58       0.78        0.73        0.78         0.88       0.64
   14     0.70     0.58       0.75        0.64        0.81         0.69       0.62
   16     0.83     0.58       0.83        0.82        0.89         0.88       0.77
   18     0.80     0.58       0.83        0.73        0.89         0.69       0.77
   20     0.84     0.60       0.87        0.82      


D rows minus world_fact (J-lens, masked, K=20)
layer balanced null p95 bal No-arm bal Yes-arm TPR No-true TPR Yes-true TNR honest
    0     0.48     0.58       0.56        0.44        0.86         0.75       0.14
    2     0.58     0.60       0.62        0.64        0.75         0.94       0.36
    4     0.47     0.58       0.44        0.45        0.89         0.81       0.07
    6     0.53     0.59       0.42        0.56        0.83         0.88       0.21
    8     0.52     0.60       0.51        0.53        0.78         0.81       0.25
   10     0.60     0.59       0.42        0.65        0.83         0.88       0.36
   12     0.56     0.59       0.54        0.55        0.83         0.81       0.29
   14     0.63     0.61       0.43        0.62        0.86         0.75       0.43
   16     0.71     0.59       0.56        0.72        0.86         0.81       0.57
   18     0.74     0.58       0.58        0.71        0.92         0.75       0.61
   20     0.72     0.59       0.58     

## 3. Transfer to the alleged arm

Fit on **all** v2 rows at each layer, apply unchanged to the alleged arm's H and D rows.
Vocabulary, standardisation and weights all come from v2. The alleged arm is 127/141
No-true, the opposite pairing from v2's lying set, so this is also a wording check:
a classifier that learned v2's phrasing lands near 0.5 here.


In [19]:
# --- 3: fit on v2, read the alleged arm ------------------------------------------
ALG = tokenfeat.TopK.load(ALG_NPZ)
ALG_EMIT = json.loads((config.RESULTS / f"slot_emitted_alleged_{cfg.lens_id}.json").read_text())
ALG_BY_ID = {i.item_id: i for i in items.load(ALLEGED_JSON, conditions=None).items}
assert ALG.layers == LAYERS, "alleged store captured on a different layer set"

A_ITEM, A_COND = np.asarray(ALG.item_ids), np.asarray(ALG.conditions)
a_truth = lambda i: ALG_BY_ID[i].meta["polarity"] == "Yes"
a_said = lambda c, i: ALG_EMIT[c][i]["p_yes"] > ALG_EMIT[c][i]["p_no"]
A_LYING = {i: (a_said("H", i) == a_truth(i)) and (a_said("D", i) != a_truth(i)) for i in set(A_ITEM)}
A_Y = np.array([(c == "D") and A_LYING[i] for i, c in zip(A_ITEM, A_COND)])
A_TRUTH = np.array([a_truth(i) for i in A_ITEM])
print(f"alleged: {len(A_Y)} rows (H, D), {int(A_Y.sum())} lie rows "
      f"({int((A_Y & ~A_TRUTH).sum())} No-true, {int((A_Y & A_TRUTH).sum())} Yes-true), "
      f"{int(((A_COND == 'D') & ~A_Y).sum())} honest D rows\n")

# Fit once per layer on every v2 row (vocab, scaling and weights from v2 only),
# apply unchanged to the alleged rows.
XFER = {}
for lens in ("j", "l"):
    for masked in (True, False):
        out = {m: [] for m in METRICS}
        for l in lens_readout._progress(LAYERS, desc=f"v2 -> alleged {lens} {'masked' if masked else 'full'}"):
            fitted = tokenfeat.fit(*STORE.at(lens, l), Y, K_PRIMARY, BANNED if masked else ())
            s = score(fitted.predict(*ALG.at(lens, l)), A_Y, A_TRUTH, A_COND)
            for m in METRICS:
                out[m].append(s[m])
        XFER[(lens, masked)] = out

viz.series_line(
    LAYERS,
    {"J-lens, answer tokens masked": XFER[("j", True)]["balanced"],
     "J-lens, full vocab": XFER[("j", False)]["balanced"],
     "logit lens, answer tokens masked": XFER[("l", True)]["balanced"],
     "logit lens, full vocab": XFER[("l", False)]["balanced"]},
    y_range=(0, 1), title="Transfer v2 -> alleged: balanced accuracy, fitted on v2 only",
    xaxis="layer", yaxis="balanced accuracy",
).show()

print(f"{'layer':>5} {'J-masked':>9} {'J-full':>7} {'L-masked':>9} {'L-full':>7} | "
      f"{'TPR No-true':>11} {'TPR Yes-true':>12} {'TNR honest D':>12}   (last three: J masked)")
for l in LAYERS[::4]:
    i = LAYERS.index(l)
    jm = XFER[("j", True)]
    print(f"{l:>5} {jm['balanced'][i]:>9.2f} {XFER[('j', False)]['balanced'][i]:>7.2f} "
          f"{XFER[('l', True)]['balanced'][i]:>9.2f} {XFER[('l', False)]['balanced'][i]:>7.2f} | "
          f"{jm['tpr_no_true'][i]:>11.2f} {jm['tpr_yes_true'][i]:>12.2f} {jm['tnr_honest_D'][i]:>12.2f}")
print("\nThe alleged arm is 127/141 No-true, so its Yes-true TPR rests on a handful of rows.")


alleged: 282 rows (H, D), 118 lie rows (112 No-true, 6 Yes-true), 23 honest D rows



v2 -> alleged j masked:   0%|          | 0/33 [00:00<?, ?it/s]

v2 -> alleged j full:   0%|          | 0/33 [00:00<?, ?it/s]

v2 -> alleged l masked:   0%|          | 0/33 [00:00<?, ?it/s]

v2 -> alleged l full:   0%|          | 0/33 [00:00<?, ?it/s]

layer  J-masked  J-full  L-masked  L-full | TPR No-true TPR Yes-true TNR honest D   (last three: J masked)
    0      0.52    0.52      0.54    0.54 |        0.04         0.00         1.00
    4      0.51    0.51      0.70    0.70 |        0.02         0.00         1.00
    8      0.85    0.85      0.76    0.76 |        0.85         0.50         0.13
   12      0.76    0.76      0.78    0.78 |        0.62         0.50         0.35
   16      0.79    0.80      0.76    0.78 |        0.69         0.33         0.30
   20      0.79    0.78      0.73    0.73 |        0.66         0.00         0.65
   24      0.69    0.68      0.82    0.77 |        0.44         0.00         0.74
   28      0.75    0.76      0.84    0.84 |        0.56         0.17         0.74
   32      0.80    0.81      0.78    0.80 |        0.65         0.17         0.78

The alleged arm is 127/141 No-true, so its Yes-true TPR rests on a handful of rows.


## 4. Which tokens carry it

At the best upper-stack layer of the **masked J-lens** curve, refit on all v2 rows and print
the fifteen highest-|weight| tokens, for the masked and full vocabularies and for both
lenses. Positive weight pushes towards "lie".


In [20]:
# --- 4: which tokens carry it ----------------------------------------------------
# l* is the argmax of the primary curve over the upper half of the fitted layers.
# It is chosen from the CV curve, not from any transfer or null number.
UPPER = [l for l in LAYERS if l >= len(LAYERS) // 2]
_bal = CV[("j", K_PRIMARY, True)]["balanced"]
L_STAR = UPPER[int(np.argmax([_bal[LAYERS.index(l)] for l in UPPER]))]
_null = NULL_P95.get(L_STAR)
print(f"l* = {L_STAR} (masked J-token CV balanced accuracy {_bal[LAYERS.index(L_STAR)]:.2f}, "
      f"null p95 {'%.2f' % _null if _null is not None else 'not run at this layer'})\n")

TOPW = {}
for lens in ("j", "l"):
    for masked in (True, False):
        fitted = tokenfeat.fit(*STORE.at(lens, L_STAR), Y, K_PRIMARY, BANNED if masked else ())
        TOPW[(lens, masked)] = fitted.top_weights(tok, n=15)
        print(f"{'J-lens' if lens == 'j' else 'logit lens'} L{L_STAR}, "
              f"{'answer tokens masked' if masked else 'full vocabulary'}, "
              f"{len(fitted.vocab)} features. Positive weight = towards 'lie'.")
        for t, w in TOPW[(lens, masked)]:
            print(f"   {w:+7.3f}  {t!r}")
        print()
print("These are the deliverable if the masked J-lens curve beat its null: a readable\n"
      "description of what the J-lens decodes at the slot while the model lies. If it did\n"
      "not, they are what a classifier reaches for when there is nothing to find.")


l* = 19 (masked J-token CV balanced accuracy 0.94, null p95 0.58)

J-lens L19, answer tokens masked, 354 features. Positive weight = towards 'lie'.
    -0.325  ' Happens'
    +0.314  ' Certainly'
    +0.270  ' Congratulations'
    +0.264  ' assurances'
    +0.244  ' Affirm'
    +0.234  ' Adequate'
    +0.231  ' Guarantee'
    +0.220  ' Thankfully'
    -0.204  ' Responder'
    +0.203  ' Doubt'
    +0.195  ' Satisf'
    +0.195  ' Maintaining'
    +0.190  ' Satisfaction'
    +0.185  ' Believe'
    +0.177  ' Ensuring'

J-lens L19, full vocabulary, 360 features. Positive weight = towards 'lie'.
    -0.319  ' Happens'
    +0.318  ' Certainly'
    +0.264  ' Congratulations'
    +0.261  ' assurances'
    +0.242  ' Affirm'
    +0.232  ' Adequate'
    +0.226  ' Guarantee'
    +0.219  ' Thankfully'
    +0.205  ' Doubt'
    -0.204  ' Responder'
    +0.196  ' Maintaining'
    +0.190  ' Satisf'
    +0.187  ' Satisfaction'
    +0.186  ' Believe'
    +0.177  ' Ensuring'

logit lens L19, answer tokens 

## 5. Summary

One JSON with every per-layer number above, so the writeup quotes a file and not a
screenshot.


In [21]:
# --- save the summary ---------------------------------------------------------
SUMMARY_JSON = config.RESULTS / f"jtoken_classifier_summary_{cfg.lens_id}.json"
r4 = lambda xs: [round(float(v), 4) for v in xs]
summary = {
    "model": cfg.name,
    "layers": LAYERS,
    "n_rows": int(len(Y)), "n_lie_rows": int(Y.sum()),
    "n_lie_rows_no_true": int((Y & ~ROW_TRUTH).sum()), "n_lie_rows_yes_true": int((Y & ROW_TRUTH).sum()),
    "k_primary": K_PRIMARY, "k_band": list(K_BAND), "k_store": STORE.k,
    "n_answer_tokens_masked": len(BANNED),
    "metric": "balanced accuracy over all 300 rows unless named otherwise",
    "cv": {f"{lens}_k{k}_{'masked' if m else 'full'}": {met: r4(v) for met, v in d.items()}
           for (lens, k, m), d in CV.items()},
    "null_p95_j_k20_masked": {str(l): round(p, 4) for l, p in NULL_P95.items()},
    "null_n": NULL_N,
    "d_only": {
        pop: {
            "n_rows": int(len(sel)), "n_lied": int(Y_D[sel].sum()), "n_honest": int((~Y_D[sel]).sum()),
            "cv": {f"{lens}_{'masked' if m else 'full'}": {met: r4(v) for met, v in d.items()}
                   for (p, lens, m), d in CV_D.items() if p == pop},
            "null_p95_j_masked": {str(l): round(v, 4) for l, v in NULL_D_P95[pop].items()},
        } for pop, sel in POPS.items()
    },
    "transfer_alleged": {f"{lens}_{'masked' if m else 'full'}": {met: r4(v) for met, v in d.items()}
                         for (lens, m), d in XFER.items()},
    "n_alleged_rows": int(len(A_Y)), "n_alleged_lie_rows": int(A_Y.sum()),
    "l_star_upper_masked_j": int(L_STAR),
    "top_weights_at_l_star": {f"{lens}_{'masked' if m else 'full'}": [[t, round(w, 3)] for t, w in tw]
                              for (lens, m), tw in TOPW.items()},
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=1))
print(f"summary -> {SUMMARY_JSON}")
print("\n`just down` syncs results/ off the box before destroying it.")


summary -> /workspace/results/jtoken_classifier_summary_gemma-3-4b-it.json

`just down` syncs results/ off the box before destroying it.
